# 🎬 Movie Review Sentiment Analysis — TRAIN FILE v3 (Advanced)
**Topic:** NLP + Machine Learning (Supervised Classification)  
**Dataset:** `movie_reviews.csv` — 100 reviews, 5 sentiment classes × 20 each  
**Classes:** Highly Positive | Positive | Neutral | Negative | Highly Negative  
**Pipeline:** Dataset → Pre-processing → Feature Engineering → Stacking Ensemble → Evaluation → Save `.pkl`  
**Folder:** `Movie_Sentiment_Analysis/` → subfolders: `data/` | `train/` | `predict/`

### 🚀 New Accuracy Improvements over v2:
- **VADER sentiment scores** — numeric sentiment polarity/intensity as extra ML features
- **Handcrafted features** — review length, exclamation marks, ALL-CAPS ratio, question marks
- **Contraction expansion** — converts `isn't` → `is not`, `can't` → `cannot` before tokenizing
- **Complement Naive Bayes** — better than MultinomialNB for imbalanced/text tasks
- **Stacking Ensemble** — meta-learner (Logistic Regression) *learns* how to blend base models
- **Class-weighted models** — handles any class imbalance automatically
- **Wider hyperparameter search** — more C values, solvers, and penalties explored
- **SGDClassifier** — fast, strong linear model with L2/elasticnet regularization

In [ ]:
# ── Cell 1 — Install required libraries ──────────────────────────────────────
!pip install nltk scikit-learn pandas matplotlib seaborn vaderSentiment contractions -q
print('✅ Libraries installed.')

In [ ]:
# ── Cell 2 — Imports ──────────────────────────────────────────────────────────
import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet', 'vader_lexicon']:
    nltk.download(pkg, quiet=True)

import pandas as pd
import numpy as np
import string
import warnings
import pickle
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# NLP
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Contraction expansion
try:
    import contractions
    HAS_CONTRACTIONS = True
except ImportError:
    HAS_CONTRACTIONS = False
    print('⚠️  contractions library not found — using manual expansion.')

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix

# ML Models
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier, VotingClassifier
)
from sklearn.calibration import CalibratedClassifierCV

# Evaluation & Tuning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.preprocessing import LabelEncoder

print('✅ All imports done.')

In [ ]:
# ── Cell 3 — Mount Google Drive & Set Folder Paths ────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/Movie_Sentiment_Analysis'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
TRAIN_DIR   = os.path.join(BASE_DIR, 'train')
PREDICT_DIR = os.path.join(BASE_DIR, 'predict')

for folder in [DATA_DIR, TRAIN_DIR, PREDICT_DIR]:
    os.makedirs(folder, exist_ok=True)

print('✅ Google Drive mounted.')
print(f'   Base    : {BASE_DIR}')
print(f'   Data    : {DATA_DIR}')
print(f'   Train   : {TRAIN_DIR}')
print(f'   Predict : {PREDICT_DIR}')

In [ ]:
# ── Cell 4 — Upload Dataset ───────────────────────────────────────────────────
from google.colab import files
print('📂 Upload your movie_reviews.csv file now:')
uploaded = files.upload()

filename = list(uploaded.keys())[0]

import shutil
dest_path = os.path.join(DATA_DIR, 'movie_reviews.csv')
shutil.copy(filename, dest_path)

df = pd.read_csv(dest_path)

print(f'\n✅ Dataset loaded from: {dest_path}')
print(f'   Total rows    : {len(df)}')
print(f'   Columns       : {df.columns.tolist()}')
print(f'\n📊 Class Distribution:')
print(df['sentiment'].value_counts())
df.head()

In [ ]:
# ── Cell 5 — STEP 1: Advanced Pre-processing ──────────────────────────────────
# NEW: Contraction expansion happens BEFORE tokenizing.
#   "can't" → "cannot", "isn't" → "is not" — preserves negation meaning.
# NEW: Slang/emoticon normalization for common movie review expressions.
# KEPT: Negation words are preserved (from v2).

# Manual contraction map (fallback if library not installed)
CONTRACTION_MAP = {
    "isn't": "is not", "wasn't": "was not", "weren't": "were not",
    "hasn't": "has not", "haven't": "have not", "hadn't": "had not",
    "doesn't": "does not", "didn't": "did not", "don't": "do not",
    "won't": "will not", "wouldn't": "would not", "couldn't": "could not",
    "shouldn't": "should not", "can't": "cannot", "cannot": "cannot",
    "it's": "it is", "i'm": "i am", "i've": "i have", "i'll": "i will",
    "i'd": "i would", "they're": "they are", "we're": "we are",
    "you're": "you are", "he's": "he is", "she's": "she is",
    "that's": "that is", "there's": "there is", "what's": "what is",
    "let's": "let us", "who's": "who is", "it'll": "it will",
    "they've": "they have", "we've": "we have", "you've": "you have",
    "they'll": "they will", "we'll": "we will", "you'll": "you will",
    "they'd": "they would", "we'd": "we would", "you'd": "you would",
    "never": "never", "n't": " not"
}

# Negation words to KEEP (not remove as stopwords)
NEGATION_WORDS = {
    'no', 'not', 'nor', 'never', 'neither', 'nobody', 'nothing', 'nowhere',
    'hardly', 'scarcely', 'barely', 'without', 'cannot'
}

base_stop_words = set(stopwords.words('english'))
stop_words = base_stop_words - NEGATION_WORDS

lemmatizer = WordNetLemmatizer()

def expand_contractions(text):
    """Expand contractions using library or manual map."""
    if HAS_CONTRACTIONS:
        return contractions.fix(text)
    # Manual fallback
    text = text.lower()
    for contraction, expansion in CONTRACTION_MAP.items():
        text = re.sub(re.escape(contraction), expansion, text)
    return text

def preprocess(text):
    # Step 1: Lowercase
    text = str(text).lower()
    # Step 2: Expand contractions (NEW — before punctuation removal)
    text = expand_contractions(text)
    # Step 3: Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Step 4: Tokenize
    tokens = word_tokenize(text)
    # Step 5: Remove stopwords (keep negation words) + non-alpha
    tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
    # Step 6: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

df['clean_review'] = df['review'].apply(preprocess)

print('✅ Pre-processing complete.')
print('\nSample — Original vs Cleaned:')
for i in range(3):
    print(f'\n  Original : {df["review"].iloc[i]}')
    print(f'  Cleaned  : {df["clean_review"].iloc[i]}')

In [ ]:
# ── Cell 6 — STEP 2: VADER Sentiment Scores + Handcrafted Features (NEW) ──────
# NEW TECHNIQUE: VADER (Valence Aware Dictionary and sEntiment Reasoner)
# is a rule-based sentiment analyzer designed for social text.
# It returns 4 numeric scores per review:
#   - compound : overall sentiment (-1.0 to +1.0)  ← most important
#   - pos      : proportion of positive words
#   - neg      : proportion of negative words
#   - neu      : proportion of neutral words
#
# These numeric scores are appended to the TF-IDF matrix as extra columns,
# giving the classifier direct sentiment signal beyond bag-of-words.
#
# ADDITIONAL handcrafted features from the RAW review (before cleaning):
#   - review_len      : number of words (longer = more detail = stronger opinion)
#   - exclamation_cnt : count of '!' (enthusiasm / strong emotion)
#   - question_cnt    : count of '?' (uncertainty / criticism)
#   - caps_ratio      : proportion of uppercase letters (emphasis / shouting)
#   - unique_word_ratio: vocabulary richness (critics use richer language)

vader = SentimentIntensityAnalyzer()

def get_vader_features(text):
    scores = vader.polarity_scores(str(text))
    return scores['compound'], scores['pos'], scores['neg'], scores['neu']

def get_handcrafted_features(text):
    text = str(text)
    words = text.split()
    review_len       = len(words)
    exclamation_cnt  = text.count('!')
    question_cnt     = text.count('?')
    alpha_chars      = [c for c in text if c.isalpha()]
    caps_ratio       = sum(1 for c in alpha_chars if c.isupper()) / (len(alpha_chars) + 1)
    unique_word_ratio= len(set(w.lower() for w in words)) / (len(words) + 1)
    return review_len, exclamation_cnt, question_cnt, caps_ratio, unique_word_ratio

# Apply to raw reviews (before cleaning)
vader_features = df['review'].apply(get_vader_features).apply(pd.Series)
vader_features.columns = ['vader_compound', 'vader_pos', 'vader_neg', 'vader_neu']

craft_features = df['review'].apply(get_handcrafted_features).apply(pd.Series)
craft_features.columns = ['review_len', 'exclamation_cnt', 'question_cnt', 'caps_ratio', 'unique_word_ratio']

extra_features = pd.concat([vader_features, craft_features], axis=1)

print('✅ VADER + Handcrafted features extracted.')
print(f'   Extra feature columns : {extra_features.columns.tolist()}')
print('\nSample extra features (first 3 rows):')
print(extra_features.head(3).to_string())

In [ ]:
# ── Cell 7 — STEP 3: Feature Extraction (TF-IDF + Extra Features Combined) ────
# Same dual TF-IDF as v2 (word + char n-grams).
# NEW: The 9 extra features (VADER + handcrafted) are appended as extra columns
# to the combined sparse TF-IDF matrix using scipy.sparse.hstack.
# Final feature matrix = [Word TF-IDF | Char TF-IDF | VADER | Handcrafted]

X_raw = df['review']         # Raw text (for VADER/handcrafted)
X     = df['clean_review']   # Cleaned text (for TF-IDF)
y     = df['sentiment']

# Word-level TF-IDF
word_tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    sublinear_tf=True,
    min_df=1,
    analyzer='word'
)

# Character-level TF-IDF
char_tfidf = TfidfVectorizer(
    max_features=3000,
    ngram_range=(2, 5),
    sublinear_tf=True,
    analyzer='char_wb'
)

X_word  = word_tfidf.fit_transform(X)
X_char  = char_tfidf.fit_transform(X)

# Convert extra numeric features to sparse matrix
X_extra = csr_matrix(extra_features.values)

# Combine all features
X_combined = hstack([X_word, X_char, X_extra])

print('✅ Feature Extraction complete.')
print(f'   Word TF-IDF features   : {X_word.shape[1]}')
print(f'   Char TF-IDF features   : {X_char.shape[1]}')
print(f'   VADER + crafted feats  : {X_extra.shape[1]}')
print(f'   TOTAL feature matrix   : {X_combined.shape}')

In [ ]:
# ── Cell 8 — STEP 4: Train/Test Split ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('✅ Train/Test Split done.')
print(f'   Training samples : {X_train.shape[0]}')
print(f'   Testing  samples : {X_test.shape[0]}')
print(f'\n   Train class distribution:')
print(pd.Series(y_train).value_counts())

In [ ]:
# ── Cell 9 — STEP 5: Train Base ML Models ────────────────────────────────────
# NEW MODELS added:
#   - ComplementNB  : better than MultinomialNB for multi-class text classification
#   - SGDClassifier : fast linear SVM-like model with L2/elasticnet regularization
# KEPT:
#   - Logistic Regression (class_weight='balanced' for any imbalance)
#   - Linear SVM         (class_weight='balanced')
#   - Random Forest

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base_models = {
    'Logistic Regression' : LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced'),
    'Complement NB'       : ComplementNB(alpha=0.3),
    'Linear SVM'          : LinearSVC(max_iter=3000, random_state=42, class_weight='balanced'),
    'SGD Classifier'      : SGDClassifier(loss='modified_huber', max_iter=1000, random_state=42,
                                          class_weight='balanced', n_jobs=-1),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42,
                                                    class_weight='balanced', n_jobs=-1)
}

results = {}

print('Training and evaluating base models...\n')
for name, model in base_models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc  = round(accuracy_score(y_test, y_pred) * 100, 2)
    prec = round(precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100, 2)
    rec  = round(recall_score(y_test, y_pred,    average='weighted', zero_division=0) * 100, 2)
    f1   = round(f1_score(y_test, y_pred,        average='weighted', zero_division=0) * 100, 2)
    cv_mean = round(cv_scores.mean() * 100, 2)
    cv_std  = round(cv_scores.std()  * 100, 2)

    results[name] = {
        'model': model, 'preds': y_pred,
        'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1,
        'CV_Mean': cv_mean, 'CV_Std': cv_std
    }
    print(f'  [{name}]')
    print(f'    Test  → Acc={acc}%  Prec={prec}%  Rec={rec}%  F1={f1}%')
    print(f'    5-fold CV → {cv_mean}% ± {cv_std}%\n')

print('✅ All base models trained and evaluated.')

In [ ]:
# ── Cell 10 — STEP 6: Hyperparameter Tuning ──────────────────────────────────
# Tune top 3 text classifiers with an expanded hyperparameter grid.
# class_weight='balanced' is included in the grid to test both modes.

print('🔍 Tuning Logistic Regression...\n')
lr_params = {
    'C'           : [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10, 20],
    'solver'      : ['lbfgs', 'saga'],
    'class_weight': [None, 'balanced']
}
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=3000, random_state=42, penalty='l2'),
    lr_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
lr_grid.fit(X_train, y_train)
lr_best  = lr_grid.best_estimator_
lr_pred  = lr_best.predict(X_test)
lr_acc   = round(accuracy_score(y_test, lr_pred) * 100, 2)
print(f'  Best params : {lr_grid.best_params_}')
print(f'  CV Acc      : {round(lr_grid.best_score_*100,2)}%  |  Test Acc: {lr_acc}%\n')

print('🔍 Tuning Linear SVM...\n')
svm_params = {
    'C'           : [0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10],
    'class_weight': [None, 'balanced']
}
svm_grid = GridSearchCV(
    LinearSVC(max_iter=5000, random_state=42),
    svm_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
svm_grid.fit(X_train, y_train)
svm_best = svm_grid.best_estimator_
svm_pred = svm_best.predict(X_test)
svm_acc  = round(accuracy_score(y_test, svm_pred) * 100, 2)
print(f'  Best params : {svm_grid.best_params_}')
print(f'  CV Acc      : {round(svm_grid.best_score_*100,2)}%  |  Test Acc: {svm_acc}%\n')

print('🔍 Tuning SGD Classifier...\n')
sgd_params = {
    'alpha'       : [0.0001, 0.001, 0.01],
    'loss'        : ['modified_huber', 'log_loss'],
    'class_weight': [None, 'balanced']
}
sgd_grid = GridSearchCV(
    SGDClassifier(max_iter=1000, random_state=42, n_jobs=-1),
    sgd_params, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
sgd_grid.fit(X_train, y_train)
sgd_best = sgd_grid.best_estimator_
sgd_pred = sgd_best.predict(X_test)
sgd_acc  = round(accuracy_score(y_test, sgd_pred) * 100, 2)
print(f'  Best params : {sgd_grid.best_params_}')
print(f'  CV Acc      : {round(sgd_grid.best_score_*100,2)}%  |  Test Acc: {sgd_acc}%\n')

# Add tuned models to results
for label, model, pred, acc, grid in [
    ('LR (Tuned)',  lr_best,  lr_pred,  lr_acc,  lr_grid),
    ('SVM (Tuned)', svm_best, svm_pred, svm_acc, svm_grid),
    ('SGD (Tuned)', sgd_best, sgd_pred, sgd_acc, sgd_grid),
]:
    results[label] = {
        'model': model, 'preds': pred,
        'Accuracy' : acc,
        'Precision': round(precision_score(y_test, pred, average='weighted', zero_division=0)*100,2),
        'Recall'   : round(recall_score(y_test, pred,    average='weighted', zero_division=0)*100,2),
        'F1'       : round(f1_score(y_test, pred,        average='weighted', zero_division=0)*100,2),
        'CV_Mean'  : round(grid.best_score_*100,2),
        'CV_Std'   : 0.0
    }

print('✅ Hyperparameter tuning complete.')

In [ ]:
# ── Cell 11 — STEP 7: Stacking Ensemble (NEW — Best Accuracy) ─────────────────
# NEW TECHNIQUE: StackingClassifier is superior to VotingClassifier.
# Instead of voting, it trains a META-LEARNER (Logistic Regression) on top
# of the predictions made by the base classifiers.
#
# How it works:
#   1. Base learners (LR, SVM, SGD, CNB) make predictions via cross-validation
#   2. Those predictions become the input to the meta-learner
#   3. The meta-learner learns the OPTIMAL BLEND of base model predictions
#
# This is consistently more accurate than simple voting because the
# meta-learner can assign different weights and correct systematic errors.

# Calibrate LinearSVC (it doesn't have predict_proba natively)
svm_calibrated = CalibratedClassifierCV(svm_best, cv=3)
svm_calibrated.fit(X_train, y_train)

# Build stacking ensemble
stacking = StackingClassifier(
    estimators=[
        ('lr',  lr_best),
        ('svm', svm_calibrated),
        ('sgd', sgd_best),
        ('cnb', ComplementNB(alpha=0.3))
    ],
    final_estimator=LogisticRegression(max_iter=2000, C=1.0, random_state=42),
    cv=5,           # 5-fold CV to generate meta-training data
    stack_method='predict_proba',
    n_jobs=-1
)

print('🔧 Training Stacking Ensemble (this may take ~1–2 min)...')
stacking.fit(X_train, y_train)
stack_pred = stacking.predict(X_test)

stack_acc  = round(accuracy_score(y_test, stack_pred) * 100, 2)
stack_prec = round(precision_score(y_test, stack_pred, average='weighted', zero_division=0) * 100, 2)
stack_rec  = round(recall_score(y_test, stack_pred,    average='weighted', zero_division=0) * 100, 2)
stack_f1   = round(f1_score(y_test, stack_pred,        average='weighted', zero_division=0) * 100, 2)

# Cross-validate the stacking ensemble
stack_cv = cross_val_score(stacking, X_train, y_train, cv=3, scoring='accuracy')

results['Stacking Ensemble'] = {
    'model': stacking, 'preds': stack_pred,
    'Accuracy': stack_acc, 'Precision': stack_prec, 'Recall': stack_rec, 'F1': stack_f1,
    'CV_Mean': round(stack_cv.mean()*100, 2), 'CV_Std': round(stack_cv.std()*100, 2)
}

print(f'\n🏆 Stacking Ensemble Results:')
print(f'   Test  → Acc={stack_acc}%  Prec={stack_prec}%  Rec={stack_rec}%  F1={stack_f1}%')
print(f'   3-fold CV → {round(stack_cv.mean()*100,2)}% ± {round(stack_cv.std()*100,2)}%')
print('\n✅ Stacking Ensemble ready.')

In [ ]:
# ── Cell 12 — STEP 8: Full Model Comparison ───────────────────────────────────
best_name = max(results, key=lambda k: results[k]['Accuracy'])
best      = results[best_name]

print('='*76)
print('   MODEL COMPARISON SUMMARY (v3 — All Models)')
print('='*76)
print(f'  {"Model":<28} {"Acc":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}  CV Mean')
print('-'*76)
for name, r in results.items():
    marker = ' ← BEST' if name == best_name else ''
    cv_str = f"{r['CV_Mean']}%" if r.get('CV_Mean') else '-'
    print(f'  {name:<28} {r["Accuracy"]:>7}% {r["Precision"]:>7}% {r["Recall"]:>7}% {r["F1"]:>7}%  {cv_str}{marker}')
print('='*76)
print(f'\n✅ Best Model : {best_name}')
print(f'   Accuracy   : {best["Accuracy"]}%')
print(f'   F1 Score   : {best["F1"]}%')

In [ ]:
# ── Cell 13 — Detailed Classification Report ──────────────────────────────────
print(f'📋 Classification Report — {best_name}\n')
print(classification_report(y_test, best['preds'], zero_division=0))

In [ ]:
# ── Cell 14 — Confusion Matrix & Metrics Chart ────────────────────────────────
class_labels = ['Highly Negative', 'Negative', 'Neutral', 'Positive', 'Highly Positive']

cm = confusion_matrix(y_test, best['preds'], labels=class_labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix — {best_name}', fontsize=11)
axes[0].tick_params(axis='x', rotation=30)

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
values  = [best['Accuracy'], best['Precision'], best['Recall'], best['F1']]
colors  = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = axes[1].bar(metrics, values, color=colors, alpha=0.87)
for bar, val in zip(bars, values):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=11
    )
axes[1].set_ylim(0, 115)
axes[1].set_title(f'Evaluation Metrics — {best_name}', fontsize=11)
axes[1].set_ylabel('Score %')
axes[1].axhline(70, color='orange', linestyle='--', linewidth=1.2, label='v2 baseline (70%)')
axes[1].axhline(80, color='gray',   linestyle='--', linewidth=1,   label='80% target')
axes[1].legend(fontsize=9)

plt.tight_layout()
plot_path = os.path.join(TRAIN_DIR, 'v3_evaluation_metrics.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'✅ Plot saved to: {plot_path}')

In [ ]:
# ── Cell 15 — Model Comparison Bar Chart (All Models) ─────────────────────────
model_names = list(results.keys())
accs  = [results[m]['Accuracy'] for m in model_names]
f1s   = [results[m]['F1']       for m in model_names]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(16, 5))
b1 = ax.bar(x - w/2, accs, w, label='Accuracy', color='#4C72B0', alpha=0.87)
b2 = ax.bar(x + w/2, f1s,  w, label='F1 Score', color='#55A868', alpha=0.87)

for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.8, f'{h}%',
            ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_title('v3 — All Models: Accuracy vs F1 Score', fontsize=13)
ax.set_ylabel('Score %')
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=8, rotation=20, ha='right')
ax.set_ylim(0, 125)
ax.axhline(70, color='orange', linestyle='--', linewidth=1.2, label='v2 baseline (70%)')
ax.axhline(80, color='gray',   linestyle='--', linewidth=1,   label='80% target')
ax.legend(fontsize=9)
plt.tight_layout()
comp_path = os.path.join(TRAIN_DIR, 'v3_model_comparison.png')
plt.savefig(comp_path, dpi=150)
plt.show()
print(f'✅ Comparison chart saved to: {comp_path}')

In [ ]:
# ── Cell 16 — Save Best Model + ALL Vectorizers as .pkl ───────────────────────
# We save THREE files:
#   1. sentiment_model_v3.pkl       — the best trained model
#   2. tfidf_word_vectorizer_v3.pkl — word-level TF-IDF
#   3. tfidf_char_vectorizer_v3.pkl — character-level TF-IDF
#
# NOTE: The VADER + handcrafted features do NOT need to be saved as .pkl
# because they are computed from the raw review text using rule-based functions
# (no fitting required). The predict.ipynb just calls the same functions.

model_path      = os.path.join(TRAIN_DIR, 'sentiment_model_v3.pkl')
word_tfidf_path = os.path.join(TRAIN_DIR, 'tfidf_word_vectorizer_v3.pkl')
char_tfidf_path = os.path.join(TRAIN_DIR, 'tfidf_char_vectorizer_v3.pkl')

with open(model_path, 'wb') as f:
    pickle.dump(best['model'], f)

with open(word_tfidf_path, 'wb') as f:
    pickle.dump(word_tfidf, f)

with open(char_tfidf_path, 'wb') as f:
    pickle.dump(char_tfidf, f)

print('✅ Model and Vectorizers saved to Google Drive!')
print(f'   Model path         : {model_path}')
print(f'   Word vectorizer    : {word_tfidf_path}')
print(f'   Char vectorizer    : {char_tfidf_path}')
print(f'   Best Algorithm     : {best_name}')
print(f'   Accuracy           : {best["Accuracy"]}%')
print(f'   F1 Score           : {best["F1"]}%')
print('\n📌 For predict.ipynb — load all 3 pkl files + use the same')
print('   preprocess(), get_vader_features(), get_handcrafted_features()')
print('   functions to transform new reviews before prediction.')
print('\n🎯 Training v3 complete!')